# Готовая Octo в симуляторе SimplerEnv (глава 3, шаг 1)

<a href="https://colab.research.google.com/github/bespatim/vla-manipulation-research/blob/chapter3/octo/notebooks/octo_simpler_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Octo** (`rail-berkeley/octo-small`, 27 млн параметров) запускается **без дообучения** на задачах
робота WidowX — того же, на котором собраны данные BridgeData V2 из главы 1. Модель получает картинку
внешней камеры и команду, отвечает непрерывными действиями захвата (как OpenVLA, но без языковой модели
на 7 млрд параметров).

Симулятор — **SimplerEnv** (ManiSkill2 + SAPIEN): сцены подогнаны под реальные кадры Bridge.
Четыре задачи, у каждой 24 фиксированные расстановки предметов:

| Задача | Команда |
|---|---|
| `widowx_spoon_on_towel` | put the spoon on the towel |
| `widowx_carrot_on_plate` | put carrot on plate |
| `widowx_stack_cube` | stack the green block on the yellow block |
| `widowx_put_eggplant_in_basket` | put eggplant into yellow basket |

Это шаг 1 главы 3: убедиться, что модель скачивается и работает в симуляторе. Шаг 2 — дообучение Octo
с датчиками суставов и без них — отдельный ноутбук.

Видеокарта: хватает бесплатной **T4**.

## 1. Проверка видеокарты

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader

## 2. Установка (~10 мин)

Отдельное окружение Python 3.10: Octo требует JAX 0.4.20, а симулятор SAPIEN 2.2 — Python не новее 3.11.
Для отрисовки сцены нужен Vulkan.

In [ ]:
BRANCH = "chapter3/octo"   # после слияния в main поменять на "main"
PY = "/content/venv-octo/bin/python"
REPO_DIR = "/content/vla-manipulation-research"
JAX_LINKS = "https://storage.googleapis.com/jax-releases/jax_cuda_releases.html"

# Vulkan для отрисовки сцены (как в Colab-примере SimplerEnv)
!apt-get install -yqq --no-install-recommends libvulkan-dev vulkan-tools > /dev/null
!mkdir -p /usr/share/vulkan/icd.d
!wget -q -O /usr/share/vulkan/icd.d/nvidia_icd.json https://raw.githubusercontent.com/haosulab/ManiSkill/main/docker/nvidia_icd.json
!wget -q -O /usr/share/glvnd/egl_vendor.d/10_nvidia.json https://raw.githubusercontent.com/haosulab/ManiSkill/main/docker/10_nvidia.json

%cd /content
!test -d vla-manipulation-research || git clone -q -b {BRANCH} https://github.com/bespatim/vla-manipulation-research.git
!cd vla-manipulation-research && git fetch -q origin && git reset -q --hard origin/{BRANCH} && git log --oneline -1
!test -d ManiSkill2_real2sim || git clone -q --depth 1 https://github.com/simpler-env/ManiSkill2_real2sim.git
!test -d SimplerEnv || git clone -q --depth 1 https://github.com/simpler-env/SimplerEnv.git
!test -d octo || (git clone -q https://github.com/octo-models/octo.git && cd octo && git checkout -q 653c54acde686fde619855f2eac0dd6edad7116b)

!pip install -q uv
!test -d /content/venv-octo || uv venv -q --python 3.10 /content/venv-octo
!uv pip install -q --python {PY} -r {REPO_DIR}/notebooks/requirements-simpler-octo.txt
!uv pip install -q --python {PY} -e /content/ManiSkill2_real2sim -e /content/SimplerEnv -e /content/octo
# JAX с поддержкой CUDA 12 (cuDNN 8.9 — под эту сборку jaxlib)
!uv pip install -q --python {PY} -f {JAX_LINKS} "jax[cuda12_pip]==0.4.20" "nvidia-cudnn-cu12==8.9.7.29"
!{PY} -c "import sys, jax, flax, tensorflow as tf, sapien; print('Python', sys.version.split()[0], '| jax', jax.__version__, jax.devices(), '| flax', flax.__version__, '| tf', tf.__version__, '| sapien', sapien.__version__)"

## 3. Первый эпизод (~3–5 мин, из них часть — скачивание модели)

Задача «морковку на тарелку», расстановка 0. Видео — с той камеры, которую получает модель.

In [ ]:
SCRIPT = f"{REPO_DIR}/scripts/simpler_octo_eval.py"
OUT = "/content/results/octo_simpler"
MODEL = "octo-small"

!cd /content/SimplerEnv && {PY} {SCRIPT} --model {MODEL} --tasks widowx_carrot_on_plate --episodes 0 --out {OUT}/first

from IPython.display import Video, display
display(Video(f"{OUT}/first/widowx_carrot_on_plate_ep00.mp4", embed=True, width=640))

## 4. Серия: 4 задачи × 24 расстановки

96 попыток, по 60 шагов (ограничение SimplerEnv для задач WidowX). Успех считается, как в SimplerEnv,
по состоянию сцены на последнем шаге.

In [ ]:
EPISODES = " ".join(str(e) for e in range(24))
!cd /content/SimplerEnv && {PY} {SCRIPT} --model {MODEL} --episodes {EPISODES} --out {OUT}/series_{MODEL}

## 5. Результаты

In [ ]:
import numpy as np
import pandas as pd

def wilson(k, n, z=1.96):
    p = k / n
    c = (p + z * z / (2 * n)) / (1 + z * z / n)
    h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / (1 + z * z / n)
    return 100 * max(c - h, 0), 100 * min(c + h, 1)

df = pd.read_csv(f"{OUT}/series_{MODEL}/summary.csv")
rows = []
for task, g in list(df.groupby("task")) + [("все задачи", df)]:
    k, n = int(g.success.sum()), len(g)
    lo, hi = wilson(k, n)
    grasp = g["final_consecutive_grasp"].mean() * 100 if "final_consecutive_grasp" in g else np.nan
    rows.append({"задача": task, "успех": f"{k}/{n} ({100 * k / n:.0f}%)", "95% ДИ": f"[{lo:.0f}%, {hi:.0f}%]",
                 "успех хоть раз за попытку": f"{int(g.ever_success.sum())}/{n}",
                 "модель, с/шаг": f"{g.infer_s.mean():.3f}", "попытка, с": f"{g.episode_s.mean():.0f}"})
pd.DataFrame(rows)

## 6. Сохранение результатов

Таблицы по шагам (`*.csv`), сводка и видео — в архив. Скачайте его или подключите Диск.

In [ ]:
!cd /content/results && zip -qr /content/results_octo_simpler.zip octo_simpler && ls -lh /content/results_octo_simpler.zip
from google.colab import files
files.download("/content/results_octo_simpler.zip")